# Notebook 33 - Second corpus, part 2: out-of-sample saliency validity and frozen 40% structures

**The one design change from the CICIoT2023 benchmark.** There, scores and harms were both computed on validation data, which the external review correctly called in-sample. Here every score is computed on training data (a natural-prior slice for Taylor and Fisher, a class-balanced training subset for semantic boundary leverage, both seed 7) and every single-channel harm is measured on the validation evaluation subsample. The two never overlap, so the score-to-harm validity check is out-of-sample. Everything else is the original benchmark's own code: `temporarily_zero_group` for the ablations, `gradient_saliency_scores` and `semantic_boundary_leverage` for the scores, `validate_score_against_harm` for the correlations, and the frozen V-C definition.

**Pre-registered claim B6** (stage 2): on the deep architecture, V-C's rank correlation with harm is at least Fisher's on at least two of the four semantic harms, with a layer-stratified bootstrap on the difference reported for both architectures. Held or not held; either is reported.

**Structures.** Realised-FLOP calibration to 40% for all five methods on both architectures (tolerance 1.5 points, minimum 8 channels per layer), frozen with raw-student audits for notebooks 34 and 35.

**Stages.** 1 bootstrap, 2 pre-registration, 3 bridge, teachers, graph, evaluation subsample and calibration material, 4 groups and scores, 5 the ablation benchmark (192 and 576 groups; resumable every 25 groups; teacher asserted unchanged afterwards), 6 validity and B6, 7 structures, 8 verdict and figures. GPU required.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, hashlib, copy, math
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.saber.bridge_iomt import build_bridge
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import (enumerate_cnn1d_channel_groups, temporarily_zero_group, prune_cnn1d_channels,
                               profile_forward_flops, count_parameters)
from src.saber.leverage import (magnitude_scores, gradient_saliency_scores, semantic_boundary_leverage,
                                validate_score_against_harm)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
IN = R / "32_iomt_bridge"
OUT = R / "33_iomt_scores_structures"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; MODEL_DIR = REPO / "models/iomt"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "B_second_corpus_scores_and_structures",
    "out_of_sample_design": ("every saliency score is computed on TRAINING data (a natural-prior slice of 40 batches, "
                             "seed 7, for Taylor and Fisher; a class-balanced training subset of up to 512 rows per "
                             "class, seed 7, for semantic boundary leverage); every single-channel harm is measured on "
                             "the VALIDATION evaluation subsample. The two never overlap, so score-to-harm validity is "
                             "out-of-sample, unlike the CICIoT2023 benchmark"),
    "scores": "random (seed 1), magnitude, Taylor, Fisher, V-C = within-layer percentile rank of raw semantic boundary "
              "leverage times layer mean Fisher; V-C definition frozen from Section 5",
    "harms": "single-group functional zeroing via temporarily_zero_group; harm_awbir (teacher-referenced, IoMT graph), "
             "harm_hsr_balanced_soc, harm_fine_macro_f1, harm_family_macro_f1, harm_benign_false_alert, all "
             "larger-is-worse relative to the teacher on the same rows",
    "claims": {
        "B6": ("on the deep architecture, V-C's Spearman correlation with harm is at least Fisher's on at least two of "
               "the four semantic harms (awbir, hsr, fine, family); reported for both architectures with a "
               "layer-stratified bootstrap of the difference (2,000 resamples)")},
    "structures": "realised-FLOP calibration to 40% for all five methods on both architectures, tolerance 1.5 points, "
                  "minimum 8 channels per layer; frozen structures and raw-student audits saved for notebooks 34 and 35",
    "no_test_access": True,
}
(OUT / "B33_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
TARGET_FLOPS = 0.40; MIN_W = 8; TOL = 0.015


In [ ]:
# Stage 3 - bridge, teachers, graph, evaluation subsample, helpers
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, taxonomy, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES)
robust_graph = pd.read_csv(IN / "asvg_edges_robust.csv")


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCHS = {"shallow": CNN1D, "deep": DeepCNN1D}
TEACHERS = {}
for arch, cls in ARCHS.items():
    ck = MODEL_DIR / f"{arch}_teacher_seed0.pt"
    payload = torch.load(ck, map_location="cpu", weights_only=False)
    m = cls(N_CLASSES); m.load_state_dict(payload["state_dict"]); TEACHERS[arch] = m.to(DEVICE).eval()
    print(arch, "teacher loaded; sha256", hashlib.sha256(ck.read_bytes()).hexdigest()[:16])

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)

# Training-data calibration material for the scores (disjoint from validation by construction)
Xt, Yt = TRAIN_LOADER.dataset.tensors
_g = torch.Generator().manual_seed(7)
_perm = torch.randperm(len(Xt), generator=_g)
CAL_NATURAL = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt[_perm[:40 * 1024]], Yt[_perm[:40 * 1024]]),
                                          batch_size=1024, shuffle=False)
_yt = Yt.numpy(); _rng7 = np.random.default_rng(7)
_bal = np.concatenate([_rng7.permutation(np.where(_yt == c)[0])[:512] for c in range(N_CLASSES) if (_yt == c).sum() > 0])
CAL_BALANCED = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt[_bal], Yt[_bal]), batch_size=1024, shuffle=False)
print("evaluation rows (validation):", len(_idx), "| score calibration rows (train): natural", 40 * 1024, "balanced", len(_bal))

_train_y = _yt
COUNTS = np.bincount(_train_y, minlength=N_CLASSES)
def class_weights(alpha):
    w = np.zeros(N_CLASSES, dtype=np.float64); nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)
TEACHER_ALPHA = {a: float(torch.load(MODEL_DIR / f"{a}_teacher_seed0.pt", map_location="cpu", weights_only=False)["alpha"]) for a in ARCHS}


def forward_logits(model, X=None):
    X = EX_X if X is None else X
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}
T_AUDIT = {a: full_model_audit(T_LOGITS[a], EX_Y, taxonomy, DEFAULT_COST_PROFILES) for a in TEACHERS}
for a in TEACHERS:
    print(a, "teacher on evaluation subsample:", {k: round(float(T_AUDIT[a][k]), 4) for k in ["fine_macro_f1", "family_macro_f1", "benign_to_attack_rate"]})


In [ ]:
# Stage 4 - channel groups and the five scores, computed on training data only
GROUPS, SCORES = {}, {}
for arch, model in TEACHERS.items():
    groups = enumerate_cnn1d_channel_groups(model, EXAMPLE_INPUT)
    GROUPS[arch] = groups
    mag = magnitude_scores(model, groups)
    lossf = nn.CrossEntropyLoss(weight=class_weights(TEACHER_ALPHA[arch]))
    grad = gradient_saliency_scores(model, CAL_NATURAL, groups, device=DEVICE, max_batches=40, criterion=lossf)
    sbl_table, _edges = semantic_boundary_leverage(model, CAL_BALANCED, groups, robust_graph, device=DEVICE,
                                                   max_samples_per_class=512, edge_weight_column="robust_weight",
                                                   normalize_by_group_size=0.0, normalize_by_flops=0.0)
    assert "sbl_raw" in sbl_table.columns, f"expected sbl_raw in {list(sbl_table.columns)}"
    sbl = sbl_table[["group_id", "sbl_raw"]].copy()
    for t in (groups, mag, grad, sbl):
        t["group_id"] = t["group_id"].astype(str)                     # merge keys must agree in type
    s = (groups[["group_id", "module_path", "channel_index"]].merge(mag, on="group_id", validate="1:1")
         .merge(grad[["group_id", "taylor", "fisher"]], on="group_id", validate="1:1").merge(sbl, on="group_id", validate="1:1"))
    assert len(s) == len(groups), f"{arch}: score merge lost rows ({len(s)} of {len(groups)})"
    s["sem_rank"] = s.groupby("module_path")["sbl_raw"].rank(pct=True)
    s["saber_v2"] = s["sem_rank"] * s["module_path"].map(s.groupby("module_path")["fisher"].mean())
    s["random"] = np.random.default_rng(1).random(len(s))
    for col in METHODS:
        assert np.isfinite(s[col]).all() and (s[col] != 0).any(), f"{arch}/{col} degenerate"
    SCORES[arch] = s; s.to_csv(OUT / f"{arch}_scores.csv", index=False)
    print(f"{arch}: {len(groups)} groups over layers {groups.groupby('module_path')['group_id'].count().to_dict()}")


In [ ]:
# Stage 5 - single-group causal ablation benchmark on the validation evaluation subsample (resumable)
for arch, model in TEACHERS.items():
    csv = OUT / f"{arch}_single_group_causal_ablation.csv"
    records = pd.read_csv(csv).to_dict("records") if csv.exists() else []
    done = {str(r["group_id"]) for r in records}
    groups = GROUPS[arch]; t_audit = T_AUDIT[arch]
    for i, group in enumerate(groups.to_dict("records"), start=1):
        gid = str(group["group_id"])
        if gid in done:
            continue
        with temporarily_zero_group(model, group):
            lg = forward_logits(model)
        a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
        aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph, weight_column="robust_weight")
        records.append({"group_id": gid, "module_path": group["module_path"], "channel_index": int(group["channel_index"]),
                        "harm_awbir": float(aw),
                        "harm_hsr_balanced_soc": float(a["hsr_balanced_soc"] - t_audit["hsr_balanced_soc"]),
                        "harm_fine_macro_f1": float(t_audit["fine_macro_f1"] - a["fine_macro_f1"]),
                        "harm_family_macro_f1": float(t_audit["family_macro_f1"] - a["family_macro_f1"]),
                        "harm_benign_false_alert": float(a["benign_to_attack_rate"] - t_audit["benign_to_attack_rate"])})
        if i % 25 == 0 or i == len(groups):
            pd.DataFrame(records).to_csv(csv, index=False); print(f"{arch}: {len(records)}/{len(groups)} groups")
    pd.DataFrame(records).to_csv(csv, index=False)
    # the teacher must be unchanged after all the temporary zeroings
    assert np.allclose(forward_logits(model), T_LOGITS[arch], atol=1e-5), f"{arch}: teacher altered by ablation"
print("ablation benchmark complete")


In [ ]:
# Stage 6 - validity: score-to-harm correlation, out of sample; B6 verdict
HARMS = ["harm_awbir", "harm_hsr_balanced_soc", "harm_fine_macro_f1", "harm_family_macro_f1"]
rng = np.random.default_rng(0)
validity, boot_rows = {}, []
for arch in TEACHERS:
    abl = pd.read_csv(OUT / f"{arch}_single_group_causal_ablation.csv"); abl["group_id"] = abl["group_id"].astype(str)
    tab = SCORES[arch].merge(abl[["group_id"] + HARMS + ["harm_benign_false_alert"]], on="group_id", validate="1:1")
    assert len(tab) == len(SCORES[arch]), f"{arch}: validity merge lost rows"
    v = validate_score_against_harm(tab, METHODS, HARMS + ["harm_benign_false_alert"])
    v.insert(0, "architecture", arch); v.to_csv(OUT / f"{arch}_validity.csv", index=False)
    rho = v.pivot(index="score", columns="harm", values="spearman")
    validity[arch] = rho
    # layer-stratified bootstrap of (V-C minus Fisher) Spearman per harm
    layers = tab["module_path"].values
    for harm in HARMS:
        diffs = []
        for _ in range(2000):
            idx = np.concatenate([rng.choice(np.where(layers == L)[0], (layers == L).sum(), replace=True) for L in np.unique(layers)])
            b = tab.iloc[idx]
            diffs.append(spearmanr(b["saber_v2"], b[harm]).statistic - spearmanr(b["fisher"], b[harm]).statistic)
        diffs = np.array(diffs)
        boot_rows.append({"architecture": arch, "harm": harm, "vc_rho": float(rho.loc["saber_v2", harm]),
                          "fisher_rho": float(rho.loc["fisher", harm]), "diff": float(rho.loc["saber_v2", harm] - rho.loc["fisher", harm]),
                          "ci_lo": float(np.percentile(diffs, 2.5)), "ci_hi": float(np.percentile(diffs, 97.5)),
                          "p_two_sided": float(2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))})
    print(f"\n{arch} Spearman(score, harm):"); print(rho.loc[METHODS, HARMS].round(3).to_string())
boot = pd.DataFrame(boot_rows); boot.to_csv(OUT / "vc_vs_fisher_bootstrap.csv", index=False)
print("\nV-C minus Fisher, layer-stratified bootstrap:"); print(boot.round(3).to_string(index=False))
deep_wins = int((boot[boot.architecture == "deep"]["diff"] >= 0).sum())
B6 = bool(deep_wins >= 2)
print("\nB6 (V-C >= Fisher on >=2 of 4 deep harms):", B6, f"({deep_wins} of 4)")


In [ ]:
# Stage 7 - frozen 40% structures for all five methods on both architectures, with raw-student audits
def removal_order(s, column):
    left = {p: int((s["module_path"] == p).sum()) for p in s["module_path"].unique()}
    seq = []
    for r in s.sort_values(column, ascending=True).itertuples():
        if left[r.module_path] - 1 < MIN_W:
            continue
        left[r.module_path] -= 1; seq.append((r.module_path, int(r.channel_index)))
    return seq


def prune_prefix(arch, seq, k):
    pm = {}
    for p, c in seq[:k]:
        pm.setdefault(p, []).append(c)
    st, _ = prune_cnn1d_channels(TEACHERS[arch], {p: sorted(cs) for p, cs in pm.items()}, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)


registry = []
for arch in TEACHERS:
    m0 = profile_forward_flops(TEACHERS[arch], EXAMPLE_INPUT)["flops_per_item"]
    realised = lambda st: 1 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / m0
    for method in METHODS:
        seq = removal_order(SCORES[arch], method)
        lo, hi = 1, len(seq)
        while lo < hi:
            mid = (lo + hi) // 2
            if realised(prune_prefix(arch, seq, mid)) >= TARGET_FLOPS: hi = mid
            else: lo = mid + 1
        st = prune_prefix(arch, seq, lo)
        rf = float(realised(st))
        pd.DataFrame([{"module_path": p, "channel_index": c} for p, c in seq[:lo]]).to_csv(OUT / f"{arch}_{method}_r40_removed_groups.csv", index=False)
        lg = forward_logits(st); a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
        aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph, weight_column="robust_weight")
        registry.append({"architecture": arch, "method": method, "target_flops": TARGET_FLOPS, "realised_flops": rf,
                         "within_tolerance": bool(abs(rf - TARGET_FLOPS) <= TOL), "removed_groups": lo, "parameters": int(count_parameters(st)),
                         "removed_groups_file": f"{arch}_{method}_r40_removed_groups.csv",
                         "raw_awbir": float(aw), "raw_fine_macro_f1": float(a["fine_macro_f1"]), "raw_family_macro_f1": float(a["family_macro_f1"]),
                         "raw_benign_to_attack_rate": float(a["benign_to_attack_rate"]), "raw_attack_to_benign_rate": float(a["attack_to_benign_rate"]),
                         "raw_hsr_balanced_soc": float(a["hsr_balanced_soc"])})
        print(f"{arch} {method:9s}: k={lo} realised={rf:.4f} params={count_parameters(st)} | raw awbir={aw:.3f} famF1={a['family_macro_f1']:.3f} b2a={a['benign_to_attack_rate']:.3f}")
reg = pd.DataFrame(registry); reg.to_csv(OUT / "structure_registry.csv", index=False)
off = reg[~reg.within_tolerance]
print("\ncalibration: mean |error|", round(float((reg.realised_flops - TARGET_FLOPS).abs().mean() * 100), 2), "pp | max",
      round(float((reg.realised_flops - TARGET_FLOPS).abs().max() * 100), 2), "pp |", "all within tolerance" if off.empty else f"OUT OF TOLERANCE: {off[['architecture','method','realised_flops']].to_dict('records')}")


In [ ]:
# Stage 8 - verdict and figures
boot = pd.read_csv(OUT / "vc_vs_fisher_bootstrap.csv"); reg = pd.read_csv(OUT / "structure_registry.csv")
verdict = {"arm": "B_second_corpus_scores_and_structures",
           "B6_vc_at_least_fisher_on_two_deep_harms": bool((boot[boot.architecture == "deep"]["diff"] >= 0).sum() >= 2),
           "deep_harms_where_vc_ge_fisher": int((boot[boot.architecture == "deep"]["diff"] >= 0).sum()),
           "shallow_harms_where_vc_ge_fisher": int((boot[boot.architecture == "shallow"]["diff"] >= 0).sum()),
           "significant_differences_95": boot[(boot.ci_lo > 0) | (boot.ci_hi < 0)][["architecture", "harm", "diff", "ci_lo", "ci_hi"]].to_dict("records"),
           "validity_spearman": {a: validity[a].loc[METHODS, HARMS].round(4).to_dict() for a in validity},
           "calibration_mean_abs_error_pp": float((reg.realised_flops - TARGET_FLOPS).abs().mean() * 100),
           "calibration_max_abs_error_pp": float((reg.realised_flops - TARGET_FLOPS).abs().max() * 100),
           "all_within_tolerance": bool(reg.within_tolerance.all()),
           "raw_students": reg[["architecture", "method", "realised_flops", "raw_awbir", "raw_family_macro_f1", "raw_benign_to_attack_rate"]].round(4).to_dict("records"),
           "prereg": json.load(open(OUT / "B33_PREREGISTRATION.json"))}
(OUT / "B33_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "validity_spearman", "raw_students")}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, arch in zip(axes, ["shallow", "deep"]):
    rho = validity[arch].loc[METHODS, HARMS]
    im = ax.imshow(rho.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(HARMS))); ax.set_xticklabels([h.replace("harm_", "") for h in HARMS], rotation=25, ha="right", fontsize=8)
    ax.set_yticks(range(len(METHODS))); ax.set_yticklabels(METHODS, fontsize=8)
    for i in range(len(METHODS)):
        for j in range(len(HARMS)):
            ax.text(j, i, f"{rho.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"IoMT {arch}: Spearman(score, harm), out of sample")
plt.colorbar(im, ax=axes, fraction=0.025); fig.savefig(OUT / "B33_validity.png", dpi=200, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7.2, 3.4))
b = boot.copy(); b["label"] = b.architecture + " / " + b.harm.str.replace("harm_", "")
ax.errorbar(range(len(b)), b["diff"], yerr=[b["diff"] - b.ci_lo, b.ci_hi - b["diff"]], fmt="o", capsize=3)
ax.axhline(0, color="0.4", lw=0.9); ax.set_xticks(range(len(b))); ax.set_xticklabels(b.label, rotation=35, ha="right", fontsize=7)
ax.set_ylabel("V-C minus Fisher (Spearman)"); ax.set_title("Layer-stratified bootstrap, 95% intervals")
fig.tight_layout(); fig.savefig(OUT / "B33_vc_vs_fisher.png", dpi=200); plt.show()
print("written ->", OUT)


In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cat ../.gitconfig > /root/.gitconfig
cat ../.git-credentials > /root/.git-credentials && chmod 600 /root/.git-credentials
python3 - <<'EOF'
import json
nb = json.load(open("notebooks/33_iomt_scores_structures.ipynb"))
for c in nb["cells"]:
    if c.get("cell_type") == "code":
        c["outputs"] = []; c["execution_count"] = None
json.dump(nb, open("/tmp/stripped.ipynb", "w"), ensure_ascii=False, indent=1)
EOF
blob=$(git hash-object -w /tmp/stripped.ipynb)
git update-index --add --cacheinfo 100644,$blob,notebooks/33_iomt_scores_structures.ipynb
git add results/saber/33_iomt_scores_structures
git commit -m "NB33 results: out-of-sample saliency validity on IoMT and frozen 40% structures. B6 FAILED: on the deep architecture V-C >= Fisher on 1 of 4 harms (AWBIR +0.025, CI crosses zero) and tied to the third decimal on the other three; Taylor marginally the best deep predictor (AWBIR 0.777). On the shallow architecture V-C > Fisher on all 4 harms, significant on AWBIR (+0.076, CI [0.023, 0.131], p=0.004), the mirror of CICIoT2023 where the significant advantage was on the deep arm. Out-of-sample validity of gradient-family scores replicates (AWBIR Spearman 0.64-0.78); magnitude negatively correlated with harm on both architectures (-0.33 to -0.53), replicating CICIoT2023. Structures: all 10 within tolerance (mean |error| 0.18 pp, max 0.54). Raw students at 40% are far less damaged than on CICIoT2023: deep Taylor/Fisher/V-C escalate 4% of benign traffic with family macro-F1 0.77-0.79 (teacher 0.81) before any retraining; shallow gradient-family raw students at 10-21%; random and magnitude destroyed on both arms"
git push origin saber-ids-method
git log --oneline -1